In [17]:
import os
import sys
from pathlib import Path
import shutil
sys.path.insert(0, '/home/mwalker/git/neoexchange/neoexchange') #point to top of file path
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'neox.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

import django
django.setup()
from django.conf import settings

import pandas as pd
import numpy as np
from IPython.display import display

from core.models import Body #Body Class
from core.models.sources import StaticSource #static class
from core.models.blocks import Block  #Block class
from core.models.frame import Frame 


os.chdir("/home/mwalker/git/neoexchange/neoexchange") #temp fix for json issue
from core.views import summarize_block_quality
from core.views import GuideMovie
from core.plots import generalized_fwhm_plotter, generalized_zeropoint_plotter  #get pngs for fwhm and zp to analyze
from astrometrics.ephem_subs import calc_moon_sep

from datetime import datetime, timedelta

In [18]:
# Query the didymos static object
didymos = Body.objects.get(name = '65803') #object.get is digano model.model call

#Query fields
ref_fields = StaticSource.objects.filter(source_type = StaticSource.REFERENCE_FIELD,
                                         superblock__groupid__startswith='65803_E10', 
                                         name__contains = '2026').order_by('id') #reference field is an attribute from the source_type attribute, gets attached via django

start = datetime(2026,7,20)
end = datetime(2026,7,22)

rows = []
for f in ref_fields:
    obs_blocks = Block.objects.filter(calibsource=f, superblock__groupid__startswith='65803_E10', 
                                      num_observed__gte=1, 
                                      block_start__gte = start,
                                      block_start__lt = end).order_by('block_start')
    for b in obs_blocks:
        rows.append({
            "field_name": f.name,
            "ra": f.ra,
            "dec": f.dec,
            "obs_date": b.block_start
        })

df = pd.DataFrame(rows)

print(df)

                   field_name          ra       dec   obs_date
0  Didymos COJ 2026 Field #14  239.013985 -22.22684 2026-07-20


In [24]:
#calculate moon separation from field 14 ---ish

obs_date = rows[0]["obs_date"]

obj_ra, obj_dec = rows[0]["ra"] , rows[0]["dec"]


moon_alt, moon_field_14_sep, moon_phase = calc_moon_sep(
    obs_date,
    obj_ra,
    obj_dec,
    "E10"
)

print(f"The approximate moon separation from {rows[0]['field_name']} is {moon_field_14_sep:.3f} deg")
print(f"{moon_phase}")

The approximate moon separation from Didymos COJ 2026 Field #14 is 20.912 deg
0.35824510158774286


In [25]:
print(obs_date)

2026-07-20 00:00:00
